# SatDiff — Kaggle training

Kaggle gives 30 GPU-hours/week, guaranteed. Colab guarantees nothing.

## One-time setup

1. **Phone-verify** at kaggle.com/settings. GPU *and* Internet are both locked
   behind it.
2. Right sidebar: **Accelerator → GPU T4 x2**. Not P100 — the T4 has fp16
   tensor cores and this config trains in mixed precision.
3. Right sidebar: **Internet → On**.
4. **Add-ons → Secrets** → `HF_TOKEN`, a **write** token from
   hf.co/settings/tokens, toggled on for this notebook.

## Why HF_TOKEN is not optional

`/kaggle/working` is wiped when the session restarts — including after an idle
timeout, which is easy to hit overnight. A completed 100-epoch run has been
lost to exactly this. The Hub is the only durable store, so cell 4 refuses to
start training when the token cannot write.

Run cells top to bottom. Every cell sets its own `cd` and `PYTHONPATH`, so a
kernel restart cannot produce a stray `No module named 'satdiff'`.

In [ ]:
# 1. GPU check. Anything but True here and nothing below works.
import torch
print("cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "Sidebar -> Accelerator -> GPU T4 x2")

In [ ]:
# 2. Code, deps, HF auth. The repo is public — cloning needs no token.
import os, sys

REPO = '/kaggle/working/satdiff-v2'

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print(f"HF_TOKEN loaded (length {len(os.environ['HF_TOKEN'])})")
except Exception as e:
    print(f'WARNING: no HF_TOKEN ({e})')
    print('Fix: Add-ons -> Secrets -> attach HF_TOKEN to THIS notebook.')

%cd /kaggle/working
# Test for .git, not just the directory: a stray `mkdir -p <repo>/checkpoints`
# leaves a non-repo directory behind, and an isdir(REPO) check would then skip
# the clone and fail later with a confusing missing-requirements.txt error.
# Checkpoints are moved aside first so a rebuild never costs a 1.1 GB refetch.
if not os.path.isdir(f'{REPO}/.git'):
    if os.path.isdir(f'{REPO}/checkpoints'):
        !mv $REPO/checkpoints /kaggle/working/_ckpt_keep
    !rm -rf $REPO
    !git clone -q https://github.com/DrKingSchultz69/satdiff-v2.git $REPO
    if os.path.isdir('/kaggle/working/_ckpt_keep'):
        !mv /kaggle/working/_ckpt_keep $REPO/checkpoints
%cd $REPO
!git pull -q

!pip install -q -r requirements.txt

os.environ['PYTHONPATH'] = f'{REPO}/src'
sys.path.insert(0, f'{REPO}/src')
print('ready')

In [ ]:
# 3. Prove the token can write BEFORE spending GPU hours. An empty or
# read-only token is the failure that loses a finished run: training appears
# to work, backs up nothing, and the session wipe takes the result with it.
import os
from huggingface_hub import HfApi

tok = os.environ.get('HF_TOKEN', '')
print('starts:', repr(tok[:3]), 'length:', len(tok))
assert tok.startswith('hf_'), 'HF_TOKEN is empty or malformed — fix the Kaggle secret'
print('account:', HfApi().whoami(token=tok)['name'])
print('token OK')

In [ ]:
# 4. Data — 94 MB, ~2 min. Splits come from the SHA-256 of each file path, not
# listdir() order, so this reproduces the identical train/val/test split as
# every other machine. That is what makes resuming on new hardware sound.
%cd /kaggle/working/satdiff-v2
!python scripts/download_data.py
!python scripts/make_splits.py

In [ ]:
# 5. OPTIONAL — seed the checkpoint from Google Drive.
#
# Only needed when the Hub has no checkpoint yet (the first Kaggle run, or
# after a lost session). Normally skip this: cell 6 pulls from the Hub by
# itself. The Drive file must be shared 'Anyone with the link' first.
#
# Pass the bare file id. Older gdown needed --fuzzy to handle a /view URL;
# current versions removed that flag and parse ids directly.
FILE_ID = '1fPimDgOtT3BjbZKV3yFm5DmkIyukAhHX'   # epoch 46

!pip install -q --upgrade gdown
!mkdir -p /kaggle/working/satdiff-v2/checkpoints
!gdown $FILE_ID -O /kaggle/working/satdiff-v2/checkpoints/last.pt
!ls -la /kaggle/working/satdiff-v2/checkpoints/
# Expect last.pt at 1101195745 bytes. Anything smaller is a truncated download
# or an HTML error page — do not train on it.

In [ ]:
# 6. Train. Refuses to start unless the Hub token can write, then pushes every
# 5 epochs and once more at the end. --resume pulls last.pt from the Hub when
# it is not on disk, so a killed session costs 5 epochs, never the run.
#
# TWO LINES TO CHECK IN THE FIRST MINUTE:
#   hub sync ON  -> DrKingSchultz69/satdiff-v1  (as ...)
#   resumed from epoch N
# Both present means it is safe to walk away. 'starting fresh' means the
# checkpoint was not found and every GPU-hour already spent is about to be
# repeated — stop it.
%cd /kaggle/working/satdiff-v2
!PYTHONPATH=src python -m satdiff.train --config configs/v1.yaml --resume

In [ ]:
# 7. The eye test. One row per class, same 4 seeds every time.
# Four identical images in a row is mode collapse, whatever KID says.
import glob
from IPython.display import Image, display

grids = sorted(glob.glob('/kaggle/working/satdiff-v2/results/grids/*.png'))
if grids:
    print(grids[-1])
    display(Image(grids[-1]))
else:
    print('none yet — the first grid lands at epoch 5')

In [ ]:
# 8. Eval: KID + CAS. Trains a ResNet-18 on real data first (~10 min), then
# scores 2,700 generated images. ~30 min total.
#
# Bars, fixed before training started (docs/eval-plan.md):
#   KID  ship <0.05   good <0.02
#   CAS  ship >=65%   good >=80%
%cd /kaggle/working/satdiff-v2
!PYTHONPATH=src python -m satdiff.eval --config configs/v1.yaml --split val

In [ ]:
# 9. Every eval run so far, newest last.
import os
import pandas as pd

csv = '/kaggle/working/satdiff-v2/results/experiments.csv'
display(pd.read_csv(csv)) if os.path.exists(csv) else print('no eval runs yet')